# Homework 3 of LLM Zoomcamp
This notebook shows my work towards completing [Homework 3](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2025/03-evaluation/homework.md) of the 2025 cohort of the course LLM Zoomcamp.

Commands to start a Qdrant Docker container:

```bash
docker pull qdrant/qdrant
docker run -p 6333:6333 -p 6334:6334 \
   -v "$(pwd)/data/qdrant:/qdrant/storage:z" \
   qdrant/qdrant
```

In [130]:
import requests

import numpy as np
import pandas as pd
from fastembed import TextEmbedding
from minsearch import Index, VectorSearch
from qdrant_client import QdrantClient, models
from rouge import Rouge
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from tqdm.auto import tqdm

In [75]:
client = QdrantClient("http://localhost:6333")

## Fetch evaluation data

In [24]:
url_prefix = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/"
docs_url = url_prefix + "search_evaluation/documents-with-ids.json"
documents = requests.get(docs_url).json()

ground_truth_url = url_prefix + "search_evaluation/ground-truth-data.csv"
df_ground_truth = pd.read_csv(ground_truth_url)
ground_truth = df_ground_truth.to_dict(orient="records")

## Define evaluation metrics

In [25]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q["document"]
        results = search_function(q)
        relevance = [d["id"] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

## Question 1: Minsearch text

Following the hit rate seen for different number of matches returned:
| Number of results | Hit rate |
|-------------------|----------|
| 5                 | 0.84     |
| 10                | 0.9      |
| 20                | 0.94     |

In [52]:
index = Index(
    text_fields=["text", "question", "section"],
    keyword_fields=["course"]
)
index.fit(documents)

In [53]:
boost_params = {"question": 1.5, "section": 0.1}

In [ ]:
q1_relevance_total = []

for q in tqdm(ground_truth):
    doc_id = q["document"]
    course = q["course"]
    results = index.search(
        q["question"],
        filter_dict={"course": course},
        boost_dict=boost_params,
        num_results=5
    )
    relevance = [d["id"] == doc_id for d in results]
    q1_relevance_total.append(relevance)

  0%|          | 0/4627 [00:00<?, ?it/s]

In [ ]:
hit_rate(q1_relevance_total)

0.848714069591528

## Embeddings

In [ ]:
q2_texts = []

for doc in documents:
    t = doc["question"]
    q2_texts.append(t)

pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)
q2_X = pipeline.fit_transform(q2_texts)

## Question 2: Vector search for question

The MRR is observed to be `0.37` which is close to `0.35` given as one of the options.

In [ ]:
vindex = VectorSearch(keyword_fields={"course"})
vindex.fit(q2_X, documents)

In [ ]:
q2_relevance_total = []

for q in tqdm(ground_truth):
    doc_id = q["document"]
    course = q["course"]
    results = vindex.search(
        pipeline.transform([q["question"]]),
        filter_dict={"course": course},
    )
    relevance = [d["id"] == doc_id for d in results]
    q2_relevance_total.append(relevance)

  0%|          | 0/4627 [00:00<?, ?it/s]

In [ ]:
mrr(q2_relevance_total)

0.3676423923074016

## Question 3: Vector search for question and answer

The hit rate seen below is `0.88` which is the closest to `0.92`.

In [71]:
q3_texts = []

for doc in documents:
    t = doc["question"] + " " + doc["text"]
    q3_texts.append(t)

In [72]:
pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)
q3_X = pipeline.fit_transform(q3_texts)
vindex = VectorSearch(keyword_fields={"course"})
vindex.fit(q3_X, documents)

In [73]:
q3_relevance_total = []

for q in tqdm(ground_truth):
    doc_id = q["document"]
    course = q["course"]
    results = vindex.search(
        pipeline.transform([q["question"]]),
        filter_dict={"course": course},
    )
    relevance = [d["id"] == doc_id for d in results]
    q3_relevance_total.append(relevance)

  0%|          | 0/4627 [00:00<?, ?it/s]

In [74]:
hit_rate(q3_relevance_total)

0.8841582018586557

## Question 4: Qdrant

MRR = `0.85`

In [92]:
collection_name = "homework_3_q4"
client.delete_collection(collection_name)
client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=512,
        distance=models.Distance.COSINE
    )
)

True

In [93]:
embedding_model = "jinaai/jina-embeddings-v2-small-en"
embedding = TextEmbedding(model_name=embedding_model)

In [ ]:
points = []
for idx, doc in enumerate(documents):
    document_embedding = list(embedding.embed(doc["question"] + " " + doc["text"]))[0]
    point = models.PointStruct(
        id=idx, 
        vector=document_embedding,
        payload={
            "text": doc["question"] + " " + doc["text"],
            "section": doc["section"],
            "course": doc["course"],
            "idx": doc["id"]
        }
    )
    points.append(point)

client.upsert(
    collection_name=collection_name,
    points=points,
    wait=True
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [117]:
def search(query: str, course: str, collection_name: str, limit: int = 1):
    results = client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model=embedding_model
        ),
        query_filter=models.Filter(
            must=[models.FieldCondition(
                key="course",
                match=models.MatchValue(value=course)
            )]
        ),
        limit=limit,
        with_payload=True
    )

    return list(results)

In [119]:
q4_relevance_total = []

for q in tqdm(ground_truth):
    doc_id = q["document"]
    course = q["course"]
    results = search(
        q["question"],
        course=course,
        collection_name=collection_name,
        limit=5
    )
    relevance = [d.payload["idx"] == doc_id for d in results[0][1]]
    q4_relevance_total.append(relevance)

  0%|          | 0/4627 [00:00<?, ?it/s]

In [120]:
mrr(q4_relevance_total)

0.8517722066133576

## Question 5: Cosine similarity

The average cosine similarity is `0.84`.

In [121]:
results_url = url_prefix + "rag_evaluation/data/results-gpt4o-mini.csv"
df_results = pd.read_csv(results_url)

In [124]:
df_results.head()

,answer_llm,answer_orig,document,question,course
0,You can sign up for the course by visiting the...,Machine Learning Zoomcamp FAQ\nThe purpose of ...,0227b872,Where can I sign up for the course?,machine-learning-zoomcamp
1,You can sign up using the link provided in the...,Machine Learning Zoomcamp FAQ\nThe purpose of ...,0227b872,Can you provide a link to sign up?,machine-learning-zoomcamp
2,"Yes, there is an FAQ for the Machine Learning ...",Machine Learning Zoomcamp FAQ\nThe purpose of ...,0227b872,Is there an FAQ for this Machine Learning course?,machine-learning-zoomcamp
3,The context does not provide any specific info...,Machine Learning Zoomcamp FAQ\nThe purpose of ...,0227b872,Does this course have a GitHub repository for ...,machine-learning-zoomcamp
4,To structure your questions and answers for th...,Machine Learning Zoomcamp FAQ\nThe purpose of ...,0227b872,How can I structure my questions and answers f...,machine-learning-zoomcamp


In [122]:
pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)

In [123]:
pipeline.fit(df_results.answer_llm + " " + df_results.answer_orig + " " + df_results.question)

,steps,"[('tfidfvectorizer', ...), ('truncatedsvd', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [128]:
llm_answer_embeddings = pipeline.transform(df_results.answer_llm)
orig_answer_embeddings = pipeline.transform(df_results.answer_orig)

In [129]:
# Normalize the embeddings
llm_norm = llm_answer_embeddings / np.linalg.norm(llm_answer_embeddings, axis=1, keepdims=True)
orig_norm = orig_answer_embeddings / np.linalg.norm(orig_answer_embeddings, axis=1, keepdims=True)

# Compute cosine similarity for each pair
cosine_similarities = np.sum(llm_norm * orig_norm, axis=1)
np.mean(cosine_similarities)

np.float64(0.8415841233490402)

## Question 6: ROUGE

The Rouge-1 F1 is `0.35`.

In [131]:
rouge_scorer = Rouge()

r = df_results.iloc[10]
scores = rouge_scorer.get_scores(r.answer_llm, r.answer_orig)[0]
scores

{'rouge-1': {'r': 0.45454545454545453,
  'p': 0.45454545454545453,
  'f': 0.45454544954545456},
 'rouge-2': {'r': 0.21621621621621623,
  'p': 0.21621621621621623,
  'f': 0.21621621121621637},
 'rouge-l': {'r': 0.3939393939393939,
  'p': 0.3939393939393939,
  'f': 0.393939388939394}}

In [132]:
rouge_scorer.get_scores(df_results.answer_llm, df_results.answer_orig, avg=True)

{'rouge-1': {'r': 0.3404359469772302,
  'p': 0.4299569796022711,
  'f': 0.3516946452113944},
 'rouge-2': {'r': 0.17516370344100232,
  'p': 0.2181134968015825,
  'f': 0.1767170469826221},
 'rouge-l': {'r': 0.3182147000427922,
  'p': 0.39908120209940684,
  'f': 0.32758565643306686}}